## XGBoost Modeling 

#### Data pre-processing steps before beginning the modeling stage

- Certain schools reported no counts of seclusion/retention and so the columns associated with these factors are blank
- The following code cells change those those factors to a count of 0 so that our model doesn't treat this data as unknown but rather as no counts to report.

In [11]:
import pandas as pd


df = pd.read_csv("../xgboost/unified_school_year_with_school_type.csv", low_memory=False)

# Indicator columns for counts
indicators = ["SCH_PHYS_IND", "SCH_MECH_IND", "SCH_SECL_IND"]
# these columns are schools where they reported no counts existed
no_counts = (df[indicators] == 0).all(axis=1)

# for the total restraint and seclusion columns for the rows in which no counts were reported but aren't unknown
for col in ["TOTAL_RESTRAINT_INSTANCES", "TOTAL_SECLUSION_INSTANCES"]:
    # replace unknown with 0
    df.loc[no_counts, col] = df.loc[no_counts, col].fillna(0)
    # keep all data where the retention and reclusion rates result in atleast 1 count or more
    
df["RS_COUNT"] = df[["TOTAL_RESTRAINT_INSTANCES", "TOTAL_SECLUSION_INSTANCES"]].sum(axis=1, min_count=1)
df = df[df["RS_COUNT"].notna()]

# training data will not involve 2023-2024 data
train = df[df["YEAR"] != "2023-2024"]
# test data will rely solely on 2023-204 data
test = df[df["YEAR"] == "2023-2024"]

### Feature selection, feature engineering, and some additional cleaning steps

#### Feature Selection 
- Remove any features that may be too indicative of restraint and seclusion rate
- Can influence the model too much and it will rely too much on these features